# Experiment: CTA Trend Walk-Forward Research Protocol


**Objective:** reproduce the locked calendar and review the committed development evidence without rerunning, retuning, or opening the contaminated candidate tail.

**Locked hypothesis:** the existing long-only CTA Trend rules can provide repeatable net risk-adjusted benefit over a constant-exposure benchmark across a preregistered broad-ETF universe. Full acceptance gates are in `docs/research-protocol.md`.


In [ ]:
# Setup: imports and reproducibility
from __future__ import annotations

import json
import sys
from pathlib import Path

import pandas as pd

SEED = 17_291
ROOT = Path.cwd()
if not (ROOT / "backend").exists():
    raise RuntimeError("Run this notebook from the repository root")
sys.path.insert(0, str(ROOT / "backend"))

from app.research import fold_manifest, load_experiment_spec, partition_candidate_holdout, walk_forward_folds
from app.store import load_bars


## Plan

1. Rebuild the locked common ETF calendar without network calls.
2. Verify the hidden 504-bar candidate tail and 14 expanding folds.
3. Read the committed, fingerprinted development evidence rather than rerunning or retuning it.
4. Apply the preregistered gates and show the recorded decision.

Planned primary metric: median net out-of-sample excess return over the constant-exposure benchmark. Planned gates: positive excess in at least 60% of folds, positive median Calmar, pooled drawdown no worse than -25%, and at least 30 closed out-of-sample trades.


In [ ]:
spec = load_experiment_spec(ROOT / "research/experiments/cta-trend-v1.json")
bars_by_symbol = {symbol: load_bars(symbol) for symbol in spec["universe"]}
assert all(not bars.empty for bars in bars_by_symbol.values())
partitions = spec["partitions"]
common_end = min(bars["date"].iloc[-1] for bars in bars_by_symbol.values())
calendar = bars_by_symbol[partitions["calendar_symbol"]]
calendar = calendar[(calendar.date >= partitions["common_history_start"]) & (calendar.date <= common_end)].reset_index(drop=True)
development, holdout = partition_candidate_holdout(calendar, holdout_bars=partitions["candidate_tail_bars"])
folds = walk_forward_folds(
    development,
    train_bars=partitions["train_bars"],
    validation_bars=partitions["validation_bars"],
    test_bars=partitions["test_bars"],
    step_bars=partitions["step_bars"],
)
{"development_bars": len(development), "folds": len(folds), "reserved_holdout": holdout}


## Results

The table below is a boundary manifest, not a performance result. Every row must satisfy `train_end < validation_start` and `validation_end < test_start`. The hidden candidate tail must not appear in any row. It remains statistically contaminated by prior inspection.


In [ ]:
manifest = fold_manifest(folds)
assert (manifest.train_end < manifest.validation_start).all()
assert (manifest.validation_end < manifest.test_start).all()
assert (manifest.test_end < holdout.start).all()
manifest


## Development decision

The committed evidence is read below. A missing test object means the locked selector found no validation survivor and therefore held cash. No candidate tail data is read.


In [ ]:
evidence = json.loads((ROOT / "output/research/cta-trend-wf-v1.json").read_text())
assert evidence["status"] == "development_complete"
assert len(evidence["folds"]) == 14
survivors = [sum(row["reject_zero_excess"] for row in fold["multiple_testing"]) for fold in evidence["folds"]]
cash_folds = sum(fold["selection"] == "cash" for fold in evidence["folds"])
decision = pd.DataFrame([
    {"gate": "validation survivors", "required": ">= 1 in a fold", "observed": f"{sum(survivors)} across 14 folds", "decision": "fail"},
    {"gate": "positive-excess test folds", "required": ">= 60%", "observed": "0/14", "decision": "fail"},
    {"gate": "closed test trades", "required": ">= 30", "observed": "0", "decision": "insufficient"},
])
assert cash_folds == 14 and not any(survivors)
decision


## Conclusion and next steps

CTA Trend v1 is rejected for insufficient validated evidence. No threshold or parameter grid is changed after seeing this result. Any revision requires a new experiment ID and attempt-ledger entry. The historical candidate tail remains contaminated and unopened by this experiment; valid confirmation requires genuinely unseen future or otherwise uninspected point-in-time data.
